# Data Understanding

This notebook validates the real ULB / Worldline credit-card fraud dataset and summarizes the modelling implications before training.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data import load_dataset

In [ ]:
dataset_path = project_root / "data" / "creditcard.csv"
transaction_dataset = load_dataset(dataset_path)

transaction_count = len(transaction_dataset)
fraud_count = int(transaction_dataset["Class"].sum())
legitimate_count = transaction_count - fraud_count
fraud_rate = fraud_count / transaction_count

summary = pd.DataFrame(
    {
        "metric": [
            "Transactions",
            "Legitimate transactions",
            "Fraudulent transactions",
            "Fraud rate",
            "Missing values",
            "Duplicate rows",
        ],
        "value": [
            transaction_count,
            legitimate_count,
            fraud_count,
            fraud_rate,
            int(transaction_dataset.isna().sum().sum()),
            int(transaction_dataset.duplicated().sum()),
        ],
    }
)
summary

In [ ]:
figure_directory = project_root / "outputs" / "figures"
figure_directory.mkdir(parents=True, exist_ok=True)

class_counts = transaction_dataset["Class"].value_counts().sort_index()
ax = class_counts.plot(kind="bar")
ax.set_title("Transaction class distribution")
ax.set_xlabel("Class")
ax.set_ylabel("Transactions")
ax.set_xticklabels(["Legitimate", "Fraud"], rotation=0)
plt.tight_layout()
plt.savefig(figure_directory / "class_distribution.png", dpi=160)
plt.show()

In [ ]:
amount_summary = transaction_dataset.groupby("Class")["Amount"].agg(
    ["count", "mean", "median", "std", "min", "max"]
)
amount_summary.index = ["Legitimate", "Fraud"]
amount_summary

In [ ]:
sampled_legitimate = transaction_dataset.loc[
    transaction_dataset["Class"] == 0, "Amount"
].sample(n=min(20_000, legitimate_count), random_state=42)

fraud_amounts = transaction_dataset.loc[
    transaction_dataset["Class"] == 1, "Amount"
]

plt.figure()
plt.hist(sampled_legitimate, bins=80, alpha=0.6, label="Legitimate sample")
plt.hist(fraud_amounts, bins=80, alpha=0.8, label="Fraud")
plt.xlim(0, transaction_dataset["Amount"].quantile(0.995))
plt.title("Transaction amount distribution")
plt.xlabel("Amount")
plt.ylabel("Transactions")
plt.legend()
plt.tight_layout()
plt.savefig(figure_directory / "amount_distribution_by_class.png", dpi=160)
plt.show()

## Modelling implications

- Fraud is extremely rare, so accuracy is not a useful primary metric.
- PR-AUC is used as the main model-comparison metric.
- Stratified train / validation / test splits are required.
- `Time` and `Amount` are scaled for linear and neural models; tree models keep original values.
- `V1`–`V28` are anonymized PCA components and are not assigned invented real-world meanings.